# TerraDS Explorer — Phase 2, Step 0

Run every cell top to bottom. This downloads **TerraDS** from Zenodo, inspects the
SQLite database, and prints the exact schema + sample rows needed to build the
graph-extraction pipeline against the **real** structure.

**After it finishes, copy ALL printed output and paste it back.**

No GitHub token needed. Download is ~335 MB.

In [ ]:
#!/usr/bin/env python3
"""
TerraDS Explorer — Phase 2, Step 0
Run this on Colab (or locally with internet). It downloads TerraDS from Zenodo,
inspects the SQLite database, and prints everything I need to build the real
graph-extraction pipeline against the ACTUAL schema (not guesses).

Just run it top to bottom and paste back the full printed output.
"""

import os, sqlite3, textwrap, glob, zipfile, tarfile, json

# ----------------------------------------------------------------------
# 1. Download TerraDS from Zenodo (record 14217386)
# ----------------------------------------------------------------------
WORK = "/content/terrads" if os.path.isdir("/content") else "./terrads"
os.makedirs(WORK, exist_ok=True)

ZENODO_RECORD = "14217386"
print(f"TerraDS Zenodo record: https://zenodo.org/records/{ZENODO_RECORD}")
print(f"Working dir: {WORK}\n")

# Fetch the file list from Zenodo's API so we don't hardcode filenames that may change.
try:
    import requests
    api = f"https://zenodo.org/api/records/{ZENODO_RECORD}"
    meta = requests.get(api, timeout=60).json()
    files = meta.get("files", [])
    print(f"Files in the Zenodo record ({len(files)}):")
    for f in files:
        key = f.get("key"); size = f.get("size", 0)
        print(f"   - {key}  ({size/1e6:.1f} MB)")
        link = f.get("links", {}).get("self")
        dest = os.path.join(WORK, key)
        if not os.path.exists(dest):
            print(f"     downloading -> {dest}")
            r = requests.get(link, timeout=600)
            open(dest, "wb").write(r.content)
        else:
            print(f"     already present -> {dest}")
except Exception as e:
    print(f"[!] Auto-download failed ({e}).")
    print("    Manually download from https://zenodo.org/records/14217386 into", WORK)

print()

# ----------------------------------------------------------------------
# 2. Unpack any archives (zip / tar) so we can see the source-code layout
# ----------------------------------------------------------------------
for arc in glob.glob(os.path.join(WORK, "*")):
    low = arc.lower()
    try:
        if low.endswith(".zip"):
            print(f"unzipping {os.path.basename(arc)} ...")
            with zipfile.ZipFile(arc) as z:
                z.extractall(os.path.join(WORK, "extracted"))
        elif low.endswith((".tar.gz", ".tgz", ".tar")):
            print(f"untarring {os.path.basename(arc)} ...")
            with tarfile.open(arc) as t:
                t.extractall(os.path.join(WORK, "extracted"))
    except Exception as e:
        print(f"  [!] could not unpack {arc}: {e}")

# ----------------------------------------------------------------------
# 3. Locate and inspect the SQLite database
# ----------------------------------------------------------------------
db_candidates = []
for root, _, fns in os.walk(WORK):
    for fn in fns:
        if fn.lower().endswith((".db", ".sqlite", ".sqlite3")):
            db_candidates.append(os.path.join(root, fn))

print("\n" + "="*70)
print("SQLITE DATABASES FOUND:", db_candidates)
print("="*70)

def inspect_db(path):
    print(f"\n### DATABASE: {path}  ({os.path.getsize(path)/1e6:.1f} MB)")
    con = sqlite3.connect(path); cur = con.cursor()
    tables = [r[0] for r in cur.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
    print("TABLES:", tables)
    for t in tables:
        print("\n" + "-"*60)
        print(f"TABLE: {t}")
        # schema
        cur.execute(f"PRAGMA table_info('{t}')")
        cols = cur.execute(f"PRAGMA table_info('{t}')").fetchall()
        print("  columns:")
        for c in cols:
            # (cid, name, type, notnull, dflt, pk)
            print(f"    {c[1]:30s} {c[2]:12s} {'PK' if c[5] else ''}")
        # row count
        try:
            n = cur.execute(f"SELECT COUNT(*) FROM '{t}'").fetchone()[0]
            print(f"  row count: {n}")
        except Exception as e:
            print(f"  row count: error {e}")
        # 2 sample rows (truncated)
        try:
            rows = cur.execute(f"SELECT * FROM '{t}' LIMIT 2").fetchall()
            colnames = [c[1] for c in cols]
            for i, row in enumerate(rows):
                print(f"  sample row {i+1}:")
                for cn, val in zip(colnames, row):
                    sval = str(val)
                    if len(sval) > 120: sval = sval[:120] + "...(truncated)"
                    print(f"      {cn}: {sval}")
        except Exception as e:
            print(f"  sample rows: error {e}")
    con.close()

for db in db_candidates:
    inspect_db(db)

# ----------------------------------------------------------------------
# 4. Show the source-code archive layout (how repos/modules are stored on disk)
# ----------------------------------------------------------------------
print("\n" + "="*70)
print("SOURCE-CODE LAYOUT (first 40 paths under 'extracted', if any)")
print("="*70)
ex = os.path.join(WORK, "extracted")
if os.path.isdir(ex):
    count = 0
    for root, dirs, fns in os.walk(ex):
        depth = root.replace(ex, "").count(os.sep)
        if depth <= 2:
            print("  " + root.replace(ex, "") or "/")
        for fn in fns[:3]:
            if fn.endswith(".tf"):
                print(f"      [.tf] {os.path.join(root, fn).replace(ex,'')}")
                count += 1
        if count > 40: break
else:
    print("  (no 'extracted' dir — source code may be inside the DB or a single archive)")

# also count total .tf files if present
tf_files = glob.glob(os.path.join(WORK, "**", "*.tf"), recursive=True)
print(f"\nTotal .tf files found on disk: {len(tf_files)}")
if tf_files:
    print("Example .tf file content (first 25 lines):")
    print(textwrap.indent("".join(open(tf_files[0], encoding='utf-8', errors='ignore').readlines()[:25]), "    "))

print("\n\n>>> DONE. Copy ALL of the above output and paste it back. <<<")
